# Khởi tạo Môi trường Nghiên cứu Khoa học trên Google Colab (GPU T4)
**Đề tài**: Ứng dụng LLM và hệ thống đa tác tử hỗ trợ ra quyết định tham khảo trong phân tích tài chính và giao dịch chứng khoán

---
### Hướng dẫn kích hoạt GPU T4 trước khi chạy:
1. Trên thanh menu Colab, chọn **Runtime** (Thời gian chạy) -> **Change runtime type** (Thay đổi loại thời gian chạy).
2. Trong mục **Hardware accelerator** (Phần cứng tăng tốc), chọn **T4 GPU**.
3. Nhấn **Save** (Lưu).
---

Notebook này hỗ trợ cả 2 cách lấy mã nguồn: **Git / GitHub (khuyên dùng)** hoặc **Google Drive**.

## Bước 1: Kiểm tra phần cứng GPU T4

In [ ]:
!nvidia-smi

## Bước 2A: Thiết lập qua Git / GitHub (KHUYÊN DÙNG)
> Chọn chạy cell này nếu bạn lưu mã nguồn dự án trên GitHub.
>
> - **Với Public Repo**: Bạn chỉ cần điền link repo.
> - **Với Private Repo**: Thêm Secret tên `GITHUB_TOKEN` vào mục Secrets (biểu tượng chìa khóa ở thanh bên trái Colab).

In [ ]:
import os
import sys

# Thay đổi thông tin repo của bạn ở đây
REPO_URL = "https://github.com/<username>/vn-llm-agents.git"  # <-- Đổi URL repo của bạn
REPO_NAME = "vn-llm-agents"

if not os.path.exists(REPO_NAME):
    # Thử lấy GITHUB_TOKEN nếu là Private Repo
    token = None
    try:
        from google.colab import userdata
        token = userdata.get('GITHUB_TOKEN')
    except Exception:
        pass

    if token:
        auth_url = REPO_URL.replace("https://", f"https://{token}@")
        !git clone {auth_url}
    else:
        !git clone {REPO_URL}

%cd {REPO_NAME}
print(f"✓ Thư mục làm việc hiện tại: {os.getcwd()}")

if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

## Bước 2B: Thiết lập qua Google Drive (DỰ PHÒNG)
> Chỉ chạy cell này nếu bạn KHÔNG dùng Git mà lưu trực tiếp code trên Google Drive.

In [ ]:
import os
import sys

try:
    from google.colab import drive
    drive.mount('/content/drive')
    project_path = '/content/drive/MyDrive/NCKH'
    if os.path.exists(project_path):
        os.chdir(project_path)
        print(f"✓ Đã chuyển thư mục làm việc về: {os.getcwd()}")
    if os.getcwd() not in sys.path:
        sys.path.insert(0, os.getcwd())
except ImportError:
    pass

## Bước 3: Cài đặt các thư viện phụ thuộc

In [ ]:
!pip install -q -U --extra-index-url https://vnstocks.com/api/packages vnstock
!pip install -q -r requirements.txt

## Bước 4: Cài đặt và Khởi chạy Ollama trên GPU T4
Tận dụng 16GB VRAM của T4 để chạy LLM cục bộ (`qwen2.5:3b` hoặc `qwen2.5:7b`), tối đa tốc độ sinh từ và không tốn phí API.

In [ ]:
# 1. Cài đặt zstd (công cụ giải nén bắt buộc của Ollama) và cài đặt Ollama
!apt-get update -qq && apt-get install -y -qq zstd
!curl -fsSL https://ollama.com/install.sh | sh

# 2. Khởi chạy tiến trình nền ollama serve
import subprocess
import time
import shutil
import requests

ollama_bin = shutil.which("ollama") or "/usr/local/bin/ollama"
print(f"Đang khởi động dịch vụ Ollama từ {ollama_bin}...")
ollama_process = subprocess.Popen([ollama_bin, "serve"])

# Chờ dịch vụ Ollama sẵn sàng kết nối
for _ in range(15):
    try:
        if requests.get("http://localhost:11434").status_code == 200:
            print("✓ Dịch vụ Ollama đã khởi động thành công!")
            break
    except Exception:
        time.sleep(1)

# 3. Tải mô hình Qwen 2.5 (3B parameters) đã cấu hình trong config.yaml
!ollama pull qwen2.5:3b
print("✓ Mô hình qwen2.5:3b đã sẵn sàng trên GPU T4!")

## Bước 5: Nạp Secret từ Colab Userdata vào Biến Môi trường
> **Cách thêm Secret trên Google Colab:**
> Click vào biểu tượng **Chìa khóa (Secrets)** ở thanh công cụ bên trái, thêm `VNSTOCK_API_KEY`, `GEMINI_API_KEY`... và bật **Notebook access**.

In [ ]:
try:
    from google.colab import userdata
    keys = ["VNSTOCK_API_KEY", "ANTHROPIC_API_KEY", "GEMINI_API_KEY", "OPENAI_API_KEY", "FIRECRAWL_API_KEY"]
    loaded_keys = []
    for key_name in keys:
        try:
            val = userdata.get(key_name)
            if val:
                os.environ[key_name] = val
                loaded_keys.append(key_name)
        except Exception:
            pass
    print(f"✓ Đã nạp secrets từ Google Colab Userdata: {loaded_keys}")
except ImportError:
    from dotenv import load_dotenv
    load_dotenv()
    print("✓ Đã nạp biến môi trường từ file .env cục bộ.")

## Bước 6: Kiểm tra tính đầy đủ của Môi trường

In [ ]:
from src.utils.check_env import check_environment
check_environment()

## Bước 7: Kiểm thử Suy luận LLM trực tiếp trên GPU T4

In [ ]:
from src.llm.client import get_llm_client_from_config

# Khởi tạo client theo cấu hình config.yaml
client = get_llm_client_from_config("agent")

test_prompt = "Hãy giải thích ngắn gọn trong 2 câu: Tỷ số P/E trong định giá cổ phiếu ngân hàng có ý nghĩa gì?"
print("Gửi prompt tới LLM (Ollama trên GPU T4)...")
response = client.generate(test_prompt, use_cache=False)

print("\n--- KẾT QUẢ TỪ MÔ HÌNH ---")
print(response)

## Bước 8: Chạy kiểm thử tự động (Unit Tests)

In [ ]:
!pytest tests/ -v

## Bước 9: Commit & Push thay đổi về GitHub (Nếu có chỉnh sửa code trên Colab)

In [ ]:
# Điền thông tin của bạn để push code về repo
!git config --global user.name "Your Name"
!git config --global user.email "your.email@example.com"

# Kiểm tra các file thay đổi
!git status

# Đẩy lên GitHub (Bỏ comment khi muốn commit)
# !git add .
# !git commit -m "Update from Google Colab GPU T4"
# !git push